imports and drive mounting

In [1]:
import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb
import wandb
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


data


In [2]:
# load the dataset from Drive
data_path = '/content/drive/MyDrive/Ketastasia/data/dataset_seq15_ready.npz'
data = np.load(data_path)

X_train, y_train = data['X_train'], data['y_train']
X_val,   y_val    = data['X_val'],   data['y_val']
X_test,  y_test   = data['X_test'],  data['y_test']

print(f"X_train shape: {X_train.shape}, X_val shape: {X_val.shape}, X_test shape: {X_test.shape}")

# read alphabetical class names from EDA JSON
with open('/content/drive/MyDrive/Ketastasia/data/eda_results/eda_stats.json') as f:
    eda_stats = json.load(f)
xgb_class_names = eda_stats['classes']
print(f"{len(xgb_class_names)} classes read from JSON: {xgb_class_names}")

# base features for naming columns
x_cols = [f'x{i}' for i in range(13)]
y_cols = [f'y{i}' for i in range(13)]
angle_cols = ['angle_left_knee', 'angle_left_elbow', 'angle_left_hip']
base_cols = x_cols + y_cols + angle_cols

X_train shape: (2940, 15, 29), X_val shape: (580, 15, 29), X_test shape: (574, 15, 29)
25 classes read from JSON: ['bench_press', 'bicep_curl', 'chest_fly', 'clean_and_jerk', 'deadlift', 'decline_bench_press', 'hammer_curl', 'hip_thrust', 'incline_bench_press', 'jump_rope', 'lat_pulldown', 'lateral_raise', 'leg_extension', 'leg_raises', 'plank', 'pullup', 'pushup', 'romanian_deadlift', 'russian_twist', 'shoulder_press', 'situp', 'squat', 't_bar_row', 'tricep_dips', 'tricep_pushdown']


LabelEncoder

In [3]:
le = LabelEncoder()

y_train_idx = le.fit_transform(y_train)
y_val_idx   = le.transform(y_val)
y_test_idx  = le.transform(y_test)

print("target encoding complete! Sample verification:")
print("Text labels:    ", y_train[:5])
print("Integer indices:", y_train_idx[:5])
print("Unique encoded classes in train:", np.unique(y_train_idx))

random_indices = np.random.choice(len(y_train), size=5, replace=False)

print("\nRandom Samples Check:")
print("Text labels:   ", y_train[random_indices])
print("Integer indices:", y_train_idx[random_indices])

target encoding complete! Sample verification:
Text labels:     ['bench_press' 'bench_press' 'bench_press' 'bench_press' 'bench_press']
Integer indices: [0 0 0 0 0]
Unique encoded classes in train: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24]

Random Samples Check:
Text labels:    ['hammer_curl' 'bicep_curl' 'shoulder_press' 'clean_and_jerk' 'squat']
Integer indices: [ 6  1 19  3 21]


feature engineering

In [4]:
def flatten_last_frame(X):
    return X[:, -1, :]

def flatten_aggregated(X):
    return np.concatenate([
        X.mean(axis=1), X.std(axis=1),
        X.min(axis=1),  X.max(axis=1)
    ], axis=1)

def flatten_full(X):
    return X.reshape(X.shape[0], -1)

def get_feature_names(feature_name, base_cols, n_timesteps=15):
    if feature_name == "aggregated":
        return ([f"mean_{c}" for c in base_cols] +
                [f"std_{c}" for c in base_cols] +
                [f"min_{c}" for c in base_cols] +
                [f"max_{c}" for c in base_cols])
    elif feature_name == "last_frame":
        return base_cols
    elif feature_name == "full_sequence":
        return [f"t{t}_{c}" for t in range(n_timesteps) for c in base_cols]
    else:
        raise ValueError(f"unknown feature_name: {feature_name}")

In [5]:
from google.colab import userdata
import wandb

wandb_api_key = userdata.get('WANDB_API_KEY_1')
wandb.login(key=wandb_api_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akeke23 (akeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
from xgboost import XGBClassifier

def run_xgb_experiment(run_name, feature_fn, feature_name, xgb_params, log_to_wandb=True):
    # Transform features based on strategy
    X_tr_feat  = feature_fn(X_train)
    X_val_feat = feature_fn(X_val)

    # Initialize and fit XGBoost Classifier (num_class=25 for your unique labels)
    model = XGBClassifier(
        random_state=42,
        n_jobs=-1,
        objective='multi:softprob',
        num_class=25,
        eval_metric='mlogloss',
        **xgb_params
    )
    model.fit(X_tr_feat, y_train_idx)

    # Calculate train metrics
    train_pred = model.predict(X_tr_feat)
    train_acc = accuracy_score(y_train_idx, train_pred)
    train_f1_macro = f1_score(y_train_idx, train_pred, average='macro')

    # Calculate validation metrics
    val_pred = model.predict(X_val_feat)
    val_acc = accuracy_score(y_val_idx, val_pred)
    val_f1_macro = f1_score(y_val_idx, val_pred, average='macro')
    val_f1_weighted = f1_score(y_val_idx, val_pred, average='weighted')

    gap = train_f1_macro - val_f1_macro

    print(f"\n=== {run_name} ===")
    print(f"Features: {feature_name} (shape: {X_tr_feat.shape[1]})")
    print(f"Params: {xgb_params}")
    print(f"Train Macro-F1: {train_f1_macro:.4f} | Val Macro-F1: {val_f1_macro:.4f} | Gap: {gap:.4f}")

    # Log results to WandB
    if log_to_wandb:
        wandb.init(
            project='ildolcefarniente',
            group='p1_xgboost',
            name=run_name,
            config={
                'model': 'xgboost',
                'feature_type': feature_name,
                'n_features': X_tr_feat.shape[1],
                **xgb_params
            },
            reinit=True
        )
        wandb.run.summary['train_accuracy'] = train_acc
        wandb.run.summary['train_f1_macro'] = train_f1_macro
        wandb.run.summary['val_accuracy'] = val_acc
        wandb.run.summary['val_f1_macro'] = val_f1_macro
        wandb.run.summary['val_f1_weighted'] = val_f1_weighted
        wandb.run.summary['gap'] = gap

        # Feature Importance Logging (uses base_cols from Cell 2)
        feature_names = get_feature_names(feature_name, base_cols)
        importances = model.feature_importances_
        top_idx = np.argsort(importances)[-15:][::-1]
        wandb.log({
            "feature_importance": wandb.plot.bar(
                wandb.Table(data=[[feature_names[i], float(importances[i])] for i in top_idx],
                            columns=["feature", "importance"]),
                "feature", "importance", title="Top 15 Feature Importances"
            )
        })

        # Confusion Matrix Logging (uses xgb_class_names from Cell 2)
        wandb.log({
            "val_confusion_matrix": wandb.plot.confusion_matrix(
                probs=None, y_true=y_val_idx.tolist(), preds=val_pred.tolist(),
                class_names=xgb_class_names
            )
        })
        wandb.finish()

    return model, val_acc, val_f1_macro, gap

In [7]:
experiments = [
    ("xgb-agg-n100-d6-lr03",           flatten_aggregated, "aggregated",
        {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.3}),
    ("xgb-agg-n500-d6-lr005",          flatten_aggregated, "aggregated",
        {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05}),
    ("xgb-agg-n300-d10-lr01",          flatten_aggregated, "aggregated",
        {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.1}),
    ("xgb-agg-n300-d4-lr01-sub08",     flatten_aggregated, "aggregated",
        {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1,
         'subsample': 0.8, 'colsample_bytree': 0.8}),
    ("xgb-agg-n100-d6-lr05",           flatten_aggregated, "aggregated",
        {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.5}),
    ("xgb-agg-n1000-d10-lr05",         flatten_aggregated, "aggregated",
        {'n_estimators': 1000, 'max_depth': 10, 'learning_rate': 0.5}),
    ("xgb-agg-n500-d15-lr03",          flatten_aggregated, "aggregated",
        {'n_estimators': 500, 'max_depth': 15, 'learning_rate': 0.3}),
    ("xgb-agg-n10-d2-lr005",           flatten_aggregated, "aggregated",
        {'n_estimators': 10, 'max_depth': 2, 'learning_rate': 0.05}),
    ("xgb-agg-n20-d3-lr001",           flatten_aggregated, "aggregated",
        {'n_estimators': 20, 'max_depth': 3, 'learning_rate': 0.01}),
    ("xgb-agg-n300-d6-lr01-reg",       flatten_aggregated, "aggregated",
        {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1,
         'subsample': 0.6, 'colsample_bytree': 0.6, 'reg_alpha': 1.0, 'reg_lambda': 2.0}),
    ("xgb-lastframe-n200-d6-lr01",     flatten_last_frame, "last_frame",
        {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}),
    ("xgb-fullseq-n300-d6-lr01",       flatten_full, "full_sequence",
        {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1}),
    ("xgb-fullseq-n500-d12-lr03",      flatten_full, "full_sequence",
        {'n_estimators': 500, 'max_depth': 12, 'learning_rate': 0.3}),
]

results_xgb = []
for run_name, feature_fn, feature_name, params in experiments:
    # Removed the trailing base_cols argument to match the definition in Cell 6
    model, val_acc, val_f1, gap = run_xgb_experiment(run_name, feature_fn, feature_name, params)
    results_xgb.append({'run': run_name, 'features': feature_name, 'params': params,
                        'val_acc': val_acc, 'val_f1_macro': val_f1, 'gap': gap, 'model': model})

# Compile results into a DataFrame and sort by validation Macro F1 score
results_xgb_df = pd.DataFrame([{k:v for k,v in r.items() if k != 'model'} for r in results_xgb])
results_xgb_df = results_xgb_df.sort_values('val_f1_macro', ascending=False)

print("\n=== XGBoost Leaderboard (Sorted by Val F1-Macro) ===")
print(results_xgb_df[['run', 'features', 'val_acc', 'val_f1_macro', 'gap']])


=== xgb-agg-n100-d6-lr03 ===
Features: aggregated (shape: 116)
Params: {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.3}
Train Macro-F1: 1.0000 | Val Macro-F1: 0.5120 | Gap: 0.4880


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


gap,0.48804
train_accuracy,1
train_f1_macro,1
val_accuracy,0.62069
val_f1_macro,0.51196
val_f1_weighted,0.62549



=== xgb-agg-n500-d6-lr005 ===
Features: aggregated (shape: 116)
Params: {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05}
Train Macro-F1: 1.0000 | Val Macro-F1: 0.5034 | Gap: 0.4966


gap,0.49658
train_accuracy,1
train_f1_macro,1
val_accuracy,0.61724
val_f1_macro,0.50342
val_f1_weighted,0.61444


...


=== xgb-fullseq-n300-d6-lr01 ===
Features: full_sequence (shape: 435)
Params: {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1}
Train Macro-F1: 1.0000 | Val Macro-F1: 0.4017 | Gap: 0.5983


gap,0.59831
train_accuracy,1
train_f1_macro,1
val_accuracy,0.54828
val_f1_macro,0.40169
val_f1_weighted,0.53031



=== xgb-fullseq-n500-d12-lr03 ===
Features: full_sequence (shape: 435)
Params: {'n_estimators': 500, 'max_depth': 12, 'learning_rate': 0.3}
Train Macro-F1: 1.0000 | Val Macro-F1: 0.3765 | Gap: 0.6235


gap,0.6235
train_accuracy,1
train_f1_macro,1
val_accuracy,0.53793
val_f1_macro,0.3765
val_f1_weighted,0.5231



=== XGBoost Leaderboard (Sorted by Val F1-Macro) ===
                           run       features   val_acc  val_f1_macro  \
9     xgb-agg-n300-d6-lr01-reg     aggregated  0.632759      0.544790   
3   xgb-agg-n300-d4-lr01-sub08     aggregated  0.637931      0.532900   
2        xgb-agg-n300-d10-lr01     aggregated  0.629310      0.520480   
6        xgb-agg-n500-d15-lr03     aggregated  0.624138      0.517555   
0         xgb-agg-n100-d6-lr03     aggregated  0.620690      0.511963   
1        xgb-agg-n500-d6-lr005     aggregated  0.617241      0.503424   
4         xgb-agg-n100-d6-lr05     aggregated  0.612069      0.474638   
5       xgb-agg-n1000-d10-lr05     aggregated  0.605172      0.469036   
11    xgb-fullseq-n300-d6-lr01  full_sequence  0.548276      0.401692   
12   xgb-fullseq-n500-d12-lr03  full_sequence  0.537931      0.376502   
7         xgb-agg-n10-d2-lr005     aggregated  0.420690      0.324221   
10  xgb-lastframe-n200-d6-lr01     last_frame  0.386207      0.268619 

In [10]:
import os
import joblib
import wandb

# ხელით ვუთითებთ ლიდერბორდის პირველ ადგილს
best_run_name = "xgb-agg-n300-d6-lr01-reg"

# სწორი ცხრილიდან (results_xgb_df) ამოგვაქვს მონაცემები
best_run_row = results_xgb_df[results_xgb_df['run'] == best_run_name].iloc[0]

# სწორი სიიდან (results_xgb) ამოგვაქვს მოდელის ობიექტი
best_xgb_obj = next(item['model'] for item in results_xgb if item['run'] == best_run_name)

print(f"Best model selected manually: {best_run_name}")
print(f"Validation F1-Macro: {best_run_row['val_f1_macro']:.4f}")

models_dir = '/content/drive/MyDrive/Ketastasia/models/'
os.makedirs(models_dir, exist_ok=True)
local_pkl_path = os.path.join(models_dir, 'pipeline1_best_xgb.pkl')
joblib.dump(best_xgb_obj, local_pkl_path)
print(f"Model saved locally at: {local_pkl_path}")

run = wandb.init(project="ildolcefarniente", job_type="model_registration", name="register_pipeline1_xgb")
artifact = wandb.Artifact(
    name="pipeline1_xgb_models",
    type="model",
    description=f"XGBoost models family. Manually selected best configuration: {best_run_name}"
)
artifact.add_file(local_pkl_path)
run.log_artifact(artifact, aliases=["latest"])
run.finish()

print("Model successfully uploaded to W&B registry.")

Best model selected manually: xgb-agg-n300-d6-lr01-reg
Validation F1-Macro: 0.5448
Model saved locally at: /content/drive/MyDrive/Ketastasia/models/pipeline1_best_xgb.pkl


Model successfully uploaded to W&B registry.
